# 03 · Recommendations on the world map
Run Notebooks 01 and 02 to prepare the observations and publish the forecasts.
Each base uses a fixed reference profile. The selected month changes the Earth values used in the comparison.
Cluster labels are shown for the month used to fit the clusters. Coordinates come from the site lookup table.

## Set up paths

In [19]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/feynmans_legacy/analog_study.py").exists()
)
sys.path.insert(0, str(ROOT / 'src'))
TERRA = ROOT / 'data/terra'
RUN = TERRA / 'analise_analogos'
RUN.mkdir(parents=True, exist_ok=True)

## Choose a month, base, and comparison profile

In [20]:
import plotly.express as px

from feynmans_legacy.analog_study import MIN_COVERAGE, profile_criteria, recommend

ANOMES = None  # None selects the latest observed month
CORPO = "lua"
BASE = None  # None selects the first base in the chosen reference profile
NICHO = "Todos"
# Minimum share of profile weight covered by data at a site. Missing criteria are
# excluded and the remaining weights are rescaled. Use 1.0 to require all criteria.
COBERTURA_MINIMA = MIN_COVERAGE


## Load the selected month

In [21]:
import json

selection = json.loads((RUN / "selecao.json").read_text(encoding="utf-8"))
latest = max(selection["anomes_observados"])
ANOMES = latest if ANOMES is None else ANOMES
if ANOMES <= latest:
    origem = "observada"
    values = pd.read_parquet(
        TERRA / "terra.parquet", filters=[("anomes", "==", ANOMES)]
    ).drop(columns="anomes")
    if values.empty:
        raise ValueError("Sem observações para o anomes selecionado.")
else:
    origem = "prevista"
    future = pd.read_parquet(
        RUN / "previsoes.parquet", filters=[("anomes", "==", ANOMES)]
    )
    if future.empty:
        raise ValueError("Sem previsões publicadas para o anomes selecionado.")
    values = future.pivot(  # noqa: PD010
        index="location_id", columns="variavel", values="valor_previsto"
    ).reset_index()
values = values.merge(
    pd.read_parquet(RUN / "estaticas.parquet"),
    on="location_id", how="left", validate="one_to_one",
)

## Apply the profile and rank sites

In [22]:
reference = pd.read_parquet(ROOT / "data" / CORPO / f"{CORPO}.parquet")
if BASE is None:
    if reference.empty:
        raise ValueError(f"Sem bases cadastradas na spec de {CORPO}.")
    BASE = reference["base_id"].iloc[0]
criteria = profile_criteria(ROOT / "config/nichos.json", CORPO, NICHO, reference, BASE)
ranking = recommend(
    values, pd.read_parquet(RUN / "coordenadas.parquet"), criteria,
    min_coverage=COBERTURA_MINIMA,
)
if ranking.empty:
    raise ValueError("Nenhum local com cobertura mínima e coordenadas para essa consulta.")


## Load cluster labels for the selected month

In [23]:
clusters = pd.read_parquet(RUN / "clusters.parquet").query("anomes == @ANOMES")
clusters = clusters[["location_id", "cluster"]]

## Check ranking coverage

In [24]:
print(f"{origem.capitalize()} | {len(ranking):,} locais elegíveis de {len(values):,} consultados")
completos = ranking["cobertura"].ge(1 - 1e-9).sum()
print(f"{completos:,} com todos os critérios; {len(ranking) - completos:,} com cobertura parcial")

Observada | 77,543 locais elegíveis de 85,726 consultados
58,283 com todos os critérios; 19,260 com cobertura parcial


## Show the five best sites per base on the world map

The map uses the selected month and comparison profile for all lunar and Martian bases. Each base has its own ranking, and a site may appear in several rankings. Hover over a site to see the base, rank, score, and coverage.

In [25]:
# Translate labels on chart copies, keeping the source data in the pipeline's schema.
body_labels = {"Lua": "Moon", "Marte": "Mars"}
base_labels = {
    "LUA_POLO_SUL": "Lunar South Pole (Shackleton rim)",
    "LUA_MARE_TUBOS": "Marius Hills lava tube",
    "MARTE_JEZERO": "Jezero Crater",
    "MARTE_GALE": "Gale Crater (Mount Sharp)",
    "MARTE_MERIDIANI": "Meridiani Planum",
    "MARTE_GELO_MEDIAS_LAT": "Northern plains (Phoenix site)",
    "MARTE_THARSIS_TUBOS": "Arsia Mons lava tubes",
    "MARTE_MEDUSAE_YARDANGS": "Medusae Fossae",
}
niche_labels = {
    "Todos": "All",
    "termico": "Thermal",
    "iluminacao_radiacao": "Lighting and radiation",
    "agua_gelo": "Water and ice",
    "agua_aridez": "Water and aridity",
    "terreno": "Terrain",
    "geologia": "Geology",
    "superficie_poeira": "Surface and dust",
    "processos": "Wind and dust processes",
    "biologia_habitabilidade": "Biology and habitability",
    "subsuperficie": "Subsurface",
    "atmosfera": "Atmosphere",
    "operacoes": "Operations",
}
source_labels = {"observada": "Observed", "prevista": "Forecast"}
plot_niche = niche_labels.get(NICHO, NICHO)
map_labels = {
    "location_id": "Earth site",
    "base_id": "Reference base",
    "corpo": "Reference body",
    "posicao": "Rank for this base",
    "nota": "Match score",
    "cobertura": "Coverage",
    "cluster": "Cluster",
    "latitude": "Latitude",
    "longitude": "Longitude",
}

LIMITE_MAPA = 5
config_path = ROOT / "config" / "nichos.json"
nichos_por_corpo = json.loads(config_path.read_text(encoding="utf-8"))
coordenadas = pd.read_parquet(RUN / "coordenadas.parquet")
colunas_mapa = ["location_id", "nota", "cobertura", "latitude", "longitude"]
partes = []
resumo_bases = []

for corpo_base, nome_corpo in (("lua", "Lua"), ("marte", "Marte")):
    perfis = pd.read_parquet(ROOT / "data" / corpo_base / f"{corpo_base}.parquet")
    for base_id in perfis["base_id"]:
        status = "Disponível"
        if NICHO != "Todos" and NICHO not in nichos_por_corpo[corpo_base]:
            status = f"Nicho indisponível para {nome_corpo}"
        else:
            criterios_base = profile_criteria(config_path, corpo_base, NICHO, perfis, base_id)
            faltantes = [
                var for var in criterios_base if var not in values or values[var].isna().all()
            ]
            if faltantes:
                status = f"Critérios sem dados no mês: {', '.join(faltantes)}"
            else:
                if corpo_base == CORPO and base_id == BASE:
                    ranking_base = ranking
                else:
                    ranking_base = recommend(
                        values[["location_id", *criterios_base]],
                        coordenadas,
                        criterios_base,
                        min_coverage=COBERTURA_MINIMA,
                    )
                if ranking_base.empty:
                    status = "Nenhum local com a cobertura mínima"
                else:
                    top = ranking_base.head(LIMITE_MAPA)[colunas_mapa].copy()
                    top["posicao"] = range(1, len(top) + 1)
                    top["base_id"] = base_id
                    top["corpo"] = nome_corpo
                    partes.append(top)
        resumo_bases.append({"base_id": base_id, "corpo": nome_corpo, "Status": status})

status_bases = pd.DataFrame(resumo_bases)
points = pd.DataFrame(columns=[*colunas_mapa, "posicao", "base_id", "corpo"])

if not partes:
    print("No recommendations to display for this selection.")
else:
    points = pd.concat(partes, ignore_index=True)
    points = points.merge(clusters, on="location_id", how="left", validate="many_to_one")
    points["cluster"] = points["cluster"].astype("string").fillna("Indisponível")
    plot_points = points.assign(
        base_id=points["base_id"].replace(base_labels),
        corpo=points["corpo"].replace(body_labels),
        cluster=points["cluster"].replace({"Indisponível": "Unavailable"}),
    )
    figure = px.scatter_geo(
        plot_points, lat="latitude", lon="longitude", color="base_id", hover_name="location_id",
        hover_data={
            "corpo": True, "posicao": True, "nota": ":.2f",
            "cobertura": ":.1%", "cluster": True, "latitude": ":.3f", "longitude": ":.3f",
        },
        labels=map_labels,
        projection="natural earth",
        title=(
            f"Top {LIMITE_MAPA} Earth sites per base · {plot_niche} · "
            f"{ANOMES} ({source_labels.get(origem, origem)})"
        ),
    )
    figure.update_traces(marker_size=9, marker_line_width=1, marker_line_color="white")
    figure.show()

## Review the highest-ranked sites

In [26]:
from IPython.display import display

LIMITE_TABELA = 1000
colunas = {
    "location_id": "Local",
    "nota": "Nota de correspondência (0–100)",
    "cobertura": "Cobertura (%)",
    "latitude": "Latitude",
    "longitude": "Longitude",
}
recomendados = ranking.head(LIMITE_TABELA)[list(colunas)].reset_index(drop=True)
recomendados.index += 1
recomendados.index.name = "Posição"
recomendados.insert(0, "Base de referência", BASE)
recomendados.insert(0, "Corpo de referência", {"lua": "Lua", "marte": "Marte"}[CORPO])
recomendados["cobertura"] = recomendados["cobertura"] * 100

display(
    recomendados.round({"nota": 2, "cobertura": 1, "latitude": 3, "longitude": 3})
    .rename(columns=colunas)
)

,Corpo de referência,Base de referência,Local,Nota de correspondência (0–100),Cobertura (%),Latitude,Longitude
Posição,,,,,,,
1,Lua,LUA_POLO_SUL,T072256,76.52,76.2,-79.75,159.75
2,Lua,LUA_POLO_SUL,T078504,74.56,76.2,-85.25,-177.75
3,Lua,LUA_POLO_SUL,T075996,74.43,76.2,-83.25,-48.25
4,Lua,LUA_POLO_SUL,T076662,74.29,76.2,-83.75,-50.75
5,Lua,LUA_POLO_SUL,T077329,74.03,76.2,-84.25,-54.75
...,...,...,...,...,...,...,...
996,Lua,LUA_POLO_SUL,T075954,64.97,76.2,-83.25,-71.75
997,Lua,LUA_POLO_SUL,T000335,64.96,90.5,81.75,58.25
998,Lua,LUA_POLO_SUL,T075029,64.96,76.2,-82.25,106.75


## Best match for each lunar and Martian base

The table and map show the first-ranked site for each base. The table also identifies bases with no recommendation for the current selection.

In [27]:
primeiros = points.loc[points["posicao"].eq(1), ["base_id", *colunas_mapa]]
tabela_bases = status_bases.merge(primeiros, on="base_id", how="left", validate="one_to_one")
tabela_bases["cobertura"] = tabela_bases["cobertura"] * 100
tabela_bases = tabela_bases.rename(columns={
    "corpo": "Corpo de referência",
    "base_id": "Base de referência",
    "location_id": "Local recomendado",
    "nota": "Nota de correspondência (0–100)",
    "cobertura": "Cobertura (%)",
    "latitude": "Latitude",
    "longitude": "Longitude",
})
display(tabela_bases.round({
    "Nota de correspondência (0–100)": 2, "Cobertura (%)": 1, "Latitude": 3, "Longitude": 3,
}))

pontos_bases = tabela_bases.loc[tabela_bases["Status"].eq("Disponível")]
if pontos_bases.empty:
    print("No recommendations to display for this selection.")
else:
    plot_bases = pontos_bases.assign(
        **{
            "Corpo de referência": pontos_bases["Corpo de referência"].replace(body_labels),
            "Base de referência": pontos_bases["Base de referência"].replace(base_labels),
        }
    )
    mapa_bases = px.scatter_geo(
        plot_bases,
        lat="Latitude",
        lon="Longitude",
        color="Corpo de referência",
        text="Base de referência",
        hover_name="Base de referência",
        hover_data={
            "Local recomendado": True,
            "Nota de correspondência (0–100)": ":.2f",
            "Cobertura (%)": ":.1f",
        },
        labels={
            "Corpo de referência": "Reference body",
            "Base de referência": "Reference base",
            "Local recomendado": "Recommended Earth site",
            "Nota de correspondência (0–100)": "Match score (0–100)",
            "Cobertura (%)": "Coverage (%)",
        },
        color_discrete_map={"Moon": "#2563eb", "Mars": "#c2410c"},
        projection="natural earth",
        title=f"Best Earth site per base · {ANOMES} · {plot_niche}",
    )
    mapa_bases.update_traces(
        textposition="top center",
        marker_size=10,
        marker_line_width=1,
        marker_line_color="white",
    )
    mapa_bases.show()

,Base de referência,Corpo de referência,Status,Local recomendado,Nota de correspondência (0–100),Cobertura (%),Latitude,Longitude
0,LUA_POLO_SUL,Lua,Disponível,T072256,76.52,76.2,-79.75,159.75
1,LUA_MARE_TUBOS,Lua,Disponível,T039384,89.02,100.0,25.75,-4.75
2,MARTE_JEZERO,Marte,Disponível,T042068,83.68,100.0,20.75,33.25
3,MARTE_GALE,Marte,Disponível,T034227,82.04,100.0,34.25,-115.25
4,MARTE_MERIDIANI,Marte,Disponível,T031402,78.66,100.0,39.25,82.25
5,MARTE_GELO_MEDIAS_LAT,Marte,Disponível,T070922,87.84,76.7,-78.75,44.25
6,MARTE_THARSIS_TUBOS,Marte,Disponível,T057810,85.40,100.0,-25.25,-68.25
7,MARTE_MEDUSAE_YARDANGS,Marte,Disponível,T036531,79.59,100.0,30.75,59.75
